# Train and serve a purchase propensity model with Signals

This notebook follows the [purchase propensity tutorial](https://docs.snowplow.io/tutorials/signals-ml-purchase-propensity/introduction). It defines a Signals attribute group, gets a point-in-time correct training table from the dataset builder, trains a model on it, and scores a live session with the same attributes. The tutorial explains each step.

By default the notebook uses the **sample data**: a training table built from the GA4 Merchandise Store sample, and a script that plays a shopper. To use your own ecommerce data in Snowflake, set `USE_SAMPLE_DATA = False` below.

## Install packages

In [ ]:
%pip install "snowplow-signals>=0.4.9" "snowflake-connector-python[pandas]" scikit-learn joblib fastapi uvicorn snowplow-tracker

## Define the model features as attributes

Add your credentials as [Colab secrets](https://x.com/GoogleColab/status/1719798406195867814): `SP_API_URL`, `SP_API_KEY`, `SP_API_KEY_ID`, and `SP_ORG_ID`, plus `SNOWPLOW_COLLECTOR_URL` for the sample data route. See [Define attributes](https://docs.snowplow.io/tutorials/signals-ml-purchase-propensity/define-attributes) for where to find them. Outside Colab, set them as environment variables instead.

The code below copies the secrets into environment variables, which the `/predict` endpoint reads later.

In [ ]:
import os

USE_SAMPLE_DATA = True  # UPDATE THIS to False to use your own data in Snowflake

try:
    from google.colab import userdata

    for name in ["SP_API_URL", "SP_API_KEY", "SP_API_KEY_ID", "SP_ORG_ID", "SNOWPLOW_COLLECTOR_URL"]:
        try:
            os.environ[name] = userdata.get(name)
        except userdata.SecretNotFoundError:
            pass
except ImportError:
    pass  # not in Colab: use the environment variables already set

In [ ]:
import os

from snowplow_signals import Signals

sp_signals = Signals(
    api_url=os.environ["SP_API_URL"],  # includes https://
    api_key=os.environ["SP_API_KEY"],
    api_key_id=os.environ["SP_API_KEY_ID"],
    org_id=os.environ["SP_ORG_ID"],
)

Define the attributes, then group them with a service.

In [ ]:
from snowplow_signals import (
    Attribute,
    Criteria,
    Criterion,
    Event,
    EventProperty,
    PageView,
)

# UPDATE THIS if your tracker sends a different version of the event
ecommerce_action = Event(
    vendor="com.snowplowanalytics.snowplow.ecommerce",
    name="snowplow_ecommerce_action",
    version="1-0-2",
)


def ecommerce_counter(name: str, action_type: str) -> Attribute:
    return Attribute(
        name=name,
        type="int32",
        events=[ecommerce_action],
        aggregation="counter",
        criteria=Criteria(
            all=[
                Criterion.eq(
                    EventProperty(
                        vendor="com.snowplowanalytics.snowplow.ecommerce",
                        name="snowplow_ecommerce_action",
                        major_version=1,
                        path="type",
                    ),
                    action_type,
                )
            ]
        ),
    )


attributes = [
    Attribute(
        name="page_views",
        type="int32",
        events=[PageView()],
        aggregation="counter",
    ),
    ecommerce_counter("product_views", "product_view"),
    ecommerce_counter("add_to_carts", "add_to_cart"),
    ecommerce_counter("remove_from_carts", "remove_from_cart"),
    ecommerce_counter("checkout_steps", "checkout_step"),
    Attribute(
        name="minutes_in_session",
        type="double",
        events=[],
        aggregation="time_since_first",
        time_unit="min",
    ),
]

In [ ]:
from snowplow_signals import Service, StreamAttributeGroup, domain_sessionid

propensity_group = StreamAttributeGroup(
    name="purchase_propensity",
    version=1,
    owner="your_email@example.com",  # UPDATE THIS
    attribute_key=domain_sessionid,
    attributes=attributes,
)

propensity_service = Service(
    name="purchase_propensity_service",
    owner="your_email@example.com",  # UPDATE THIS
    attribute_groups=[propensity_group],
)

Test the group on the last hour of events in your warehouse. This needs a warehouse connection in Signals, so skip it with the sample data if you don't have one.

In [ ]:
if not USE_SAMPLE_DATA:
    display(sp_signals.test(attribute_group=propensity_group))

In [ ]:
published = sp_signals.publish([propensity_group, propensity_service])
print(f"{len(published)} objects published")

## Build the training dataset

See [Build the dataset](https://docs.snowplow.io/tutorials/signals-ml-purchase-propensity/build-training-dataset) for what each row means. With your own data, the next cells submit a dataset run in your Snowflake account and wait for it. With the sample data, they're skipped.

In [ ]:
if not USE_SAMPLE_DATA:
    propensity_group = sp_signals.get_attribute_group("purchase_propensity")
    print(f"{propensity_group.name} v{propensity_group.version}")

    from datetime import datetime, timezone

    from snowplow_signals import Criteria, Criterion, EventProperty, TrainingSpan, WarehouseTable

    purchase = Criteria(
        all=[
            Criterion.eq(
                EventProperty(
                    vendor="com.snowplowanalytics.snowplow.ecommerce",
                    name="snowplow_ecommerce_action",
                    major_version=1,
                    path="type",
                ),
                "transaction",
            )
        ]
    )

    run = sp_signals.submit_dataset_run_with_session_anchors(
        attribute_groups=[propensity_group],
        goal_criteria=purchase,
        training_span=TrainingSpan(
            # UPDATE THIS to a window where you have ecommerce events
            start_time=datetime(2026, 8, 1, tzinfo=timezone.utc),
            end_time=datetime(2026, 10, 1, tzinfo=timezone.utc),
        ),
        dataset_table=WarehouseTable(table="purchase_propensity_dataset"),
    )
    print(run.id)

In [ ]:
if not USE_SAMPLE_DATA:
    import time

    from snowplow_signals import DatasetRunStatus

    while True:
        status = sp_signals.get_dataset_run_status(run.id)
        if status.status == DatasetRunStatus.SUCCESS:
            break
        if status.status == DatasetRunStatus.FAILED:
            raise RuntimeError("Dataset run failed")
        time.sleep(5)

    dataset = status.dataset
    print(f"{dataset.database}.{dataset.schema_}.{dataset.table}")

Load the training table. With your own data, set the `SNOWFLAKE_` variables for the Snowflake connector first.

In [ ]:
if USE_SAMPLE_DATA:
    import pandas as pd

    df = pd.read_csv(
        "https://raw.githubusercontent.com/snowplow/documentation/main/static/tutorial-assets/signals-ml-purchase-propensity/purchase_propensity_sample.csv.gz",
        parse_dates=["anchor_ts"],
    )
else:
    import snowflake.connector

    conn = snowflake.connector.connect(
        account=os.environ["SNOWFLAKE_ACCOUNT"],
        user=os.environ["SNOWFLAKE_USER"],
        password=os.environ["SNOWFLAKE_PASSWORD"],
        warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    )
    cursor = conn.cursor()
    cursor.execute(f"SELECT * FROM {dataset.database}.{dataset.schema_}.{dataset.table}")
    df = cursor.fetch_pandas_all()
    conn.close()

    # Snowflake returns unquoted column names in uppercase
    df.columns = df.columns.str.lower()

df.head()

## Train and evaluate the model

The feature preparation goes in its own module, `features.py`, which the `/predict` endpoint also imports. See [Train the model](https://docs.snowplow.io/tutorials/signals-ml-purchase-propensity/train-model).

In [ ]:
%%writefile features.py
import pandas as pd

FEATURES = [
    "page_views",
    "product_views",
    "add_to_carts",
    "remove_from_carts",
    "checkout_steps",
    "minutes_in_session",
]
COUNTERS = [f for f in FEATURES if f != "minutes_in_session"]


def to_features(rows: pd.DataFrame) -> pd.DataFrame:
    """Turn Signals attribute values into model inputs."""
    X = rows.reindex(columns=FEATURES).astype("float64")
    X[COUNTERS] = X[COUNTERS].fillna(0)
    return X

In [ ]:
from features import FEATURES, to_features

session_start = df.groupby("domain_sessionid")["anchor_ts"].transform("min")
cutoff = session_start.quantile(0.8)

train = df[session_start < cutoff]
test = df[session_start >= cutoff]

X_train, y_train = to_features(train), train["label"]
X_test, y_test = to_features(test), test["label"]

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

model = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)

Evaluate on the held-out sessions.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

scores = model.predict_proba(X_test)[:, 1]
print(f"ROC AUC:           {roc_auc_score(y_test, scores):.3f}")
print(f"Average precision: {average_precision_score(y_test, scores):.3f}")
print(f"Positive rate:     {y_test.mean():.3f}")

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, scores)

Train the final model on all the rows and save it.

In [ ]:
import joblib

final_model = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=42)
final_model.fit(to_features(df), df["label"])
joblib.dump(final_model, "propensity_model.joblib")

## Score live sessions

The `/predict` endpoint fetches a session's live attributes from Signals and scores them. See [Score live sessions](https://docs.snowplow.io/tutorials/signals-ml-purchase-propensity/serve-predictions). Write it to `app.py` and start it in the background:

In [ ]:
%%writefile app.py
import os

import joblib
import pandas as pd
from fastapi import FastAPI
from snowplow_signals import Signals

from features import FEATURES, to_features

model = joblib.load("propensity_model.joblib")
sp_signals = Signals(
    api_url=os.environ["SP_API_URL"],
    api_key=os.environ["SP_API_KEY"],
    api_key_id=os.environ["SP_API_KEY_ID"],
    org_id=os.environ["SP_ORG_ID"],
)

app = FastAPI()


@app.get("/predict")
def predict(domain_sessionid: str):
    attributes = sp_signals.get_service_attributes(
        name="purchase_propensity_service",
        attribute_key="domain_sessionid",
        identifier=domain_sessionid,
    )
    X = to_features(pd.DataFrame([attributes]))
    score = float(model.predict_proba(X)[0, 1])
    return {
        "domain_sessionid": domain_sessionid,
        "score": round(score, 3),
        "attributes": {name: attributes.get(name) for name in FEATURES},
    }

In [ ]:
import subprocess, time

server = subprocess.Popen(["uvicorn", "app:app", "--port", "8000"], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
time.sleep(5)

With the sample data, run the shopper script. It sends a short shopping journey to your Collector for a new session, and prints the score after each step.

In [ ]:
if USE_SAMPLE_DATA:
    !curl -sO https://raw.githubusercontent.com/snowplow/documentation/main/static/tutorial-assets/signals-ml-purchase-propensity/shopper.py
    !python shopper.py

With your own data, browse your website in a private window, find your `domain_sessionid` in the Snowplow Inspector, and request a score:

In [ ]:
import requests

domain_sessionid = "YOUR_DOMAIN_SESSIONID"  # UPDATE THIS
if not USE_SAMPLE_DATA:
    print(requests.get("http://localhost:8000/predict", params={"domain_sessionid": domain_sessionid}).json())

Stop the endpoint when you're done:

In [ ]:
server.terminate()